In [ ]:
"""
stage6_downstream_augmentation.py
============================================================================
Kaggle Notebook. GPU required.

Stage 6 of the thesis plan (RQ-D): train DenseNet-121 classifiers on
real training data augmented with diffusion reconstructions, comparing
UNFILTERED augmentation against UNCERTAINTY-FILTERED augmentation.

WHAT THE "SYNTHETIC" IMAGES ACTUALLY ARE -- state this in your methods.
Stage 1 inverted each REAL train image to its latent z_T and regenerated it
20 times; the saved "{image_id}__mean" array is the ensemble mean, i.e. an
approximate RECONSTRUCTION of an image already in the training set, not a
novel sample. So this experiment tests whether uncertainty-filtered
diffusion RECONSTRUCTIONS improve a downstream classifier -- a narrower
(and more defensible) claim than "synthetic data helps".

THREE TRAINING MODES (all identical except which reconstructions are added):
  "none"           -- real images only. The baseline the plan omits but that
                      you need: without it, neither augmented model can be
                      shown to help at all.
  "all"            -- real + EVERY reconstruction            (plan's Model 1)
  "filtered"       -- real + only LOW-variance reconstructions (plan's Model 2)
  "random_matched" -- real + a RANDOM subset of reconstructions, sized to
                      exactly match "filtered".

Why random_matched matters: "all" and "filtered" differ in BOTH selection
and dataset size, so if "filtered" wins you cannot tell whether uncertainty
filtering helped or whether a smaller/cleaner set helped. "random_matched"
holds size constant so only the selection criterion varies. Running it turns
a confounded result into a clean one for the same compute.

SPLIT DISCIPLINE (per the thesis plan's contamination requirement):
  - Only split=="train" images are ever used for training or augmentation.
  - The validation set is REAL images only, and is IDENTICAL across all
    modes (fixed seed), so model selection is comparable.
  - split=="test" is NEVER touched here. That is Stage 7.

OUTPUT: one checkpoint dir per mode under CHECKPOINT_ROOT, plus
stage6_training_summary.csv comparing final/best val loss across modes.
"""

# ============================================================================
# CELL 0 — Imports
# ============================================================================
# NOTE: do NOT unconditionally `!pip install` here. Kaggle already ships
# h5py, torch, numpy and pandas, and an unnecessary pip install can resolve
# dependencies in a way that breaks an already-loaded torch, producing:
#   AttributeError: partially initialized module 'torch' has no attribute 'fx'
# That error survives for the rest of the session -- it needs a KERNEL
# RESTART, not a code change. The guard below installs only if genuinely
# missing, and tells you to restart if it had to.
try:
    import h5py
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "h5py"], check=True)
    raise SystemExit(
        "h5py was missing and has been installed. RESTART THE KERNEL "
        "(Run > Restart & Clear Cell Outputs), then re-run this cell."
    )

import json
import time
import warnings
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from tqdm.auto import tqdm


# ============================================================================
# CELL 1 — CONFIG
# ============================================================================
H5_PATH = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_512.h5"
METADATA_CSV_PATH = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_metadata.csv"

TARGET_CLASSES = ["Pneumothorax", "Consolidation", "Nodule/Mass", "Cardiomegaly", "Atelectasis"]

# Stage 1 BASELINE variance-map output per class. Each value may be a single
# directory, or a LIST of directories for a class whose generation spanned
# several Kaggle sessions (e.g. Cardiomegaly). These supply BOTH the
# reconstruction images ("__mean") and the uncertainty used for filtering
# ("__variance"), so no separate synthetic-image dataset is needed.
STAGE1_DIRS_BY_CLASS = {
    "Pneumothorax": "/kaggle/input/datasets/pgc17ms072/pneumothorax-uncertainity-train-control",
    "Consolidation": "/kaggle/input/datasets/pgc17ms072/consolidation-gen-train",
    "Atelectasis": "/kaggle/input/datasets/kartikichandratre/atelectasis-uncertainity-train-eta1-n20-g1",
    "Nodule/Mass": "/kaggle/input/datasets/pgc17ms072/nodule-uncertainity-train-eta1-n20-g1",
    # "Consolidation": "/kaggle/input/consolidation-gen-train",
    "Cardiomegaly": [
        "/kaggle/input/datasets/pgc17ms072/cardiomegaly-uncertainity-train-eta1-n20-g1-sess0",
        "/kaggle/input/datasets/pgc17ms072/cardiomegaly-unceratainty-train-eta1-n20-g1-sess1",
         "/kaggle/input/datasets/kartikichandratre/cardiomelgaly-uncertainity-sess2",
    ],
}

CHECKPOINT_ROOT = "/kaggle/working/stage6_models"
OUTPUT_DIR = "/kaggle/working"

# Which modes to train, in order. "none" first: it is the reference both
# augmented models must beat for RQ-D to have a positive answer.
MODES = ["none", "all", "filtered", "random_matched"]

# Keep reconstructions whose variance_mean is at or below this percentile.
# Computed PER CLASS, not globally: different pathologies can sit at
# different variance scales, and a global cut would silently drop whole
# classes rather than the noisiest images within each.
VARIANCE_PERCENTILE = 75.0

# --- Class-imbalance handling ----------------------------------------------
# Positive prevalence in this dataset spans ~1.5% (Pneumothorax) to ~52%
# (Cardiomegaly). With plain BCE, a rare head can minimise loss by predicting
# near-zero everywhere. pos_weight scales the POSITIVE term of each class's
# loss by (negatives / positives), counteracting that.
#
# TWO NON-NEGOTIABLE RULES, both easy to get wrong:
#
# 1. The weights are computed from the REAL training images ONLY, never from
#    each arm's own augmented set. Augmentation changes class counts, and
#    UNEVENLY (Cardiomegaly nearly doubles; Pneumothorax adds far less), so
#    per-arm weights would give the four arms DIFFERENT loss functions and
#    destroy the controlled comparison. One vector, computed once, applied
#    to every arm.
# 2. Every arm in a comparison must share the same setting. Never compare a
#    weighted arm against an unweighted one -- RUN_TAG below keeps the two
#    sets of checkpoints separate so they cannot be mixed by accident.
USE_POS_WEIGHT = True

# Raw negative/positive ratios can be extreme (~65x for a 1.5%-prevalence
# class), which destabilises training and pushes the model to over-predict
# rare classes. Capping trades some rare-class recall for stability. Set to
# None for uncapped. Whatever you choose is a documented hyperparameter.
POS_WEIGHT_CAP = 20.0

# Appended to every checkpoint dir and output filename, so a weighted run and
# an unweighted run coexist instead of silently overwriting each other.
RUN_TAG = "weighted" if USE_POS_WEIGHT else "unweighted"

IMG_SIZE = 512
MODEL_INPUT_SIZE = 512
DROP_RATE = 0.3            # kept identical to Stage 4 for architectural consistency
BATCH_SIZE = 8
ACCUMULATION_STEPS = 2     # effective batch 16
USE_AMP = True
NUM_WORKERS = 4
NUM_EPOCHS = 12
LEARNING_RATE = 1e-4
VAL_FRACTION = 0.15
RANDOM_SEED = 42           # fixes the real train/val split -- MUST be identical across modes

# Seed for TRAINING stochasticity only: weight init, batch shuffling, dropout
# masks. Deliberately separate from RANDOM_SEED so that varying it changes
# ONLY the training run, while the train/val split and the augmentation
# selection stay byte-identical. That is what makes a multi-seed comparison
# measure training variance rather than data variance.
TRAINING_SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================================
# CELL 2 — Build the reconstruction manifest from Stage 1 output
# ============================================================================
def index_stage1_dirs(dirs):
    """{image_id: npz_path} across one or several directories."""
    dirs = [dirs] if isinstance(dirs, (str, Path)) else list(dirs)
    index = {}
    for d in dirs:
        d = Path(d)
        if not d.exists():
            raise FileNotFoundError(f"Stage 1 directory not found: {d}")
        for p in sorted(d.glob("*.npz")):
            with np.load(p) as data:
                for key in data.files:
                    if key.endswith("__variance"):
                        index[key[: -len("__variance")]] = p
    if not index:
        raise RuntimeError(f"No '*__variance' keys found under {dirs}")
    return index


def build_reconstruction_manifest(stage1_dirs_by_class):
    """
    One row per available reconstruction: image_id, finding, npz_path,
    variance_mean. variance_mean is read from the companion _meta.csv where
    present (cheap) and computed from the array otherwise (slower but exact).
    """
    rows = []
    for class_name, dirs in stage1_dirs_by_class.items():
        index = index_stage1_dirs(dirs)

        # Prefer the companion _meta.csv -- avoids loading 512x512 arrays.
        meta_lookup = {}
        dir_list = [dirs] if isinstance(dirs, (str, Path)) else list(dirs)
        for d in dir_list:
            for mp in sorted(Path(d).glob("*_meta.csv")):
                mdf = pd.read_csv(mp)
                if {"image_id", "variance_mean"}.issubset(mdf.columns):
                    meta_lookup.update(dict(zip(mdf["image_id"].astype(str),
                                                 mdf["variance_mean"])))

        n_computed = 0
        for image_id, npz_path in tqdm(index.items(), desc=f"manifest: {class_name}", leave=False):
            variance_mean = meta_lookup.get(image_id)
            if variance_mean is None:
                with np.load(npz_path) as data:
                    variance_mean = float(data[f"{image_id}__variance"].mean())
                n_computed += 1
            rows.append({
                "image_id": image_id, "finding": class_name,
                "npz_path": str(npz_path), "variance_mean": float(variance_mean),
            })
        print(f"  {class_name}: {len(index)} reconstruction(s)"
              + (f" ({n_computed} variance_mean computed from arrays)" if n_computed else ""))

    manifest = pd.DataFrame(rows)

    # An image can carry several findings and so appear in several class
    # cohorts. Keep ONE row per image_id -- otherwise it would be augmented
    # into the training set multiple times, silently over-weighting it.
    before = len(manifest)
    manifest = manifest.sort_values("variance_mean").drop_duplicates(subset="image_id", keep="first")
    if before != len(manifest):
        print(f"  de-duplicated {before - len(manifest)} multi-class reconstruction row(s)")

    return manifest.reset_index(drop=True)


def select_augmentation_ids(manifest, mode, percentile, seed):
    """Returns (list_of_image_ids, description) for the given mode."""
    if mode == "none":
        return [], "real images only (no augmentation)"

    if mode == "all":
        return manifest["image_id"].tolist(), f"all {len(manifest)} reconstructions"

    # Per-class percentile cut on variance_mean (low variance = kept).
    keep = []
    for class_name, group in manifest.groupby("finding"):
        cutoff = np.percentile(group["variance_mean"], percentile)
        kept = group[group["variance_mean"] <= cutoff]
        keep.append(kept)
        print(f"    {class_name}: keeping {len(kept)}/{len(group)} "
              f"(variance_mean <= {cutoff:.6f})")
    filtered = pd.concat(keep)

    if mode == "filtered":
        return (filtered["image_id"].tolist(),
                f"{len(filtered)} low-variance reconstructions (<= p{percentile:g} per class)")

    if mode == "random_matched":
        # Same COUNT as "filtered", drawn at random -- isolates the effect of
        # the selection criterion from the effect of dataset size.
        rng = np.random.default_rng(seed)
        sampled = manifest.sample(n=len(filtered), random_state=int(rng.integers(1 << 31)))
        return (sampled["image_id"].tolist(),
                f"{len(sampled)} randomly-chosen reconstructions (size-matched to 'filtered')")

    raise ValueError(f"Unknown mode: {mode!r}")


# ============================================================================
# CELL 3 — Dataset serving real images (h5) and reconstructions (npz)
# ============================================================================
class RealPlusReconstructionDataset(Dataset):
    """
    Serves multi-label chest X-rays from two sources:
      - source "real":  (512,512) float32 in [-1,1] from the h5 file
      - source "recon": the "{image_id}__mean" array from a Stage 1 npz

    Labels always come from the metadata for that image_id -- a
    reconstruction inherits its source image's labels.
    """
    def __init__(self, records, h5_path, target_classes, model_input_size=MODEL_INPUT_SIZE):
        self.records = records            # list of dicts: image_id, source, npz_path, labels
        self.h5_path = h5_path
        self.target_classes = target_classes
        self.model_input_size = model_input_size
        self._h5file = None

    def _h5(self):
        if self._h5file is None:
            self._h5file = h5py.File(self.h5_path, "r")
        return self._h5file

    def __len__(self):
        return len(self.records)

    def _to_tensor(self, arr):
        t = torch.from_numpy(np.ascontiguousarray(arr, dtype=np.float32))[None, None]
        if t.shape[-1] != self.model_input_size:
            t = F.interpolate(t, size=(self.model_input_size, self.model_input_size),
                               mode="bilinear", align_corners=False)
        return t.squeeze(0).repeat(3, 1, 1)

    def __getitem__(self, idx):
        rec = self.records[idx]
        if rec["source"] == "real":
            arr = self._h5()[rec["image_id"]][()]
        else:
            with np.load(rec["npz_path"]) as data:
                arr = data[f"{rec['image_id']}__mean"]

        target = torch.zeros(len(self.target_classes), dtype=torch.float32)
        for i, cls in enumerate(self.target_classes):
            if cls in rec["labels"]:
                target[i] = 1.0
        return self._to_tensor(arr), target, rec["image_id"]


def build_records(metadata, image_ids, source, npz_lookup=None):
    labels_by_id = dict(zip(metadata["image_id"].astype(str), metadata["labels"]))
    records = []
    for image_id in image_ids:
        if image_id not in labels_by_id:
            continue
        records.append({
            "image_id": image_id, "source": source,
            "npz_path": None if source == "real" else npz_lookup[image_id],
            "labels": labels_by_id[image_id],
        })
    return records


# ============================================================================
# CELL 4 — Model + training (mirrors Stage 4: AMP, accumulation, resumable)
# ============================================================================
def build_densenet121(num_classes, drop_rate=DROP_RATE, pretrained=True):
    weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.densenet121(weights=weights, drop_rate=drop_rate)
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(nn.Dropout(p=drop_rate), nn.Linear(in_features, num_classes))
    return model


def compute_pos_weight(metadata, real_train_ids, target_classes, cap=POS_WEIGHT_CAP):
    """
    Per-class pos_weight = negatives / positives, computed from the REAL
    training images only.

    Deliberately ignores augmentation: reconstructions are added unevenly
    across classes, so deriving weights per-arm would hand the four arms
    different objectives and confound the very comparison this experiment
    exists to make. One vector, shared by every arm.

    Returns (tensor of shape [n_classes], stats DataFrame for the record).
    """
    labels_by_id = dict(zip(metadata["image_id"].astype(str), metadata["labels"]))
    rows = []
    for i, class_name in enumerate(target_classes):
        n_pos = sum(1 for image_id in real_train_ids
                    if class_name in labels_by_id.get(image_id, []))
        n_neg = len(real_train_ids) - n_pos
        if n_pos == 0:
            # No positives: any weight is meaningless, and dividing would blow
            # up. Weight 1.0 leaves the head at plain BCE.
            raw = weight = 1.0
            warnings.warn(f"'{class_name}' has NO positive examples in the real "
                           f"training set — pos_weight set to 1.0 for that head.")
        else:
            raw = n_neg / n_pos
            weight = min(raw, cap) if cap is not None else raw
        rows.append({"class_name": class_name, "n_positives": n_pos,
                     "n_negatives": n_neg,
                     "prevalence": n_pos / max(len(real_train_ids), 1),
                     "raw_pos_weight": raw, "applied_pos_weight": weight,
                     "capped": bool(cap is not None and raw > cap)})

    stats = pd.DataFrame(rows)
    return torch.tensor(stats["applied_pos_weight"].values, dtype=torch.float32), stats


def train_one_model(mode, train_records, val_records, h5_path, checkpoint_dir,
                     pos_weight=None):
    """Trains one mode to completion, resumable. Returns a summary dict."""
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = checkpoint_dir / "checkpoint.pt"
    best_path = checkpoint_dir / "best.pt"

    train_ds = RealPlusReconstructionDataset(train_records, h5_path, TARGET_CLASSES)
    val_ds = RealPlusReconstructionDataset(val_records, h5_path, TARGET_CLASSES)
    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=pin,
                               persistent_workers=NUM_WORKERS > 0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=pin,
                             persistent_workers=NUM_WORKERS > 0)

    # Within one run tag, every mode starts from the SAME initialization: the
    # only difference between models must be the augmentation data. Across
    # seed-replication runs this is what is varied, deliberately.
    torch.manual_seed(TRAINING_SEED)
    model = build_densenet121(len(TARGET_CLASSES), DROP_RATE, pretrained=True).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight.to(DEVICE) if pos_weight is not None else None)
    # Validation loss is deliberately UNWEIGHTED even when training is
    # weighted: it is used to select checkpoints and to compare arms, so it
    # must measure the same quantity regardless of the training objective.
    # A weighted val loss would not be comparable to an unweighted run's.
    val_criterion = nn.BCEWithLogitsLoss()

    amp_enabled = bool(USE_AMP and DEVICE.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    start_epoch, best_val_loss, best_epoch = 0, float("inf"), -1
    history = []   # per-epoch record; persisted in the checkpoint so a
                   # resumed run keeps the curve from before the restart
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ck["model_state"])
        optimizer.load_state_dict(ck["optimizer_state"])
        if amp_enabled and ck.get("scaler_state") is not None:
            scaler.load_state_dict(ck["scaler_state"])
        start_epoch = ck["epoch"] + 1
        best_val_loss = ck.get("best_val_loss", float("inf"))
        best_epoch = ck.get("best_epoch", -1)
        history = ck.get("history", [])
        print(f"  resumed at epoch {start_epoch}/{NUM_EPOCHS} "
              f"({len(history)} epoch(s) of history recovered)")

    for epoch in range(start_epoch, NUM_EPOCHS):
        model.train()
        running, seen, t0 = 0.0, 0, time.time()
        optimizer.zero_grad(set_to_none=True)
        for step, (images, targets, _) in enumerate(
                tqdm(train_loader, desc=f"[{mode}] epoch {epoch+1}/{NUM_EPOCHS}", leave=False)):
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                loss = criterion(model(images), targets) / ACCUMULATION_STEPS
            scaler.scale(loss).backward()
            if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
            running += loss.item() * ACCUMULATION_STEPS * images.size(0)
            seen += images.size(0)
        train_loss = running / max(seen, 1)

        model.eval()
        vrunning, vseen = 0.0, 0
        with torch.no_grad():
            for images, targets, _ in val_loader:
                images = images.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)
                with torch.amp.autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                    loss = val_criterion(model(images), targets)
                vrunning += loss.item() * images.size(0)
                vseen += images.size(0)
        val_loss = vrunning / max(vseen, 1)

        is_best = val_loss < best_val_loss
        if is_best:
            best_val_loss, best_epoch = val_loss, epoch + 1
            torch.save(model.state_dict(), best_path)

        epoch_minutes = (time.time() - t0) / 60
        history.append({
            "mode": mode, "epoch": epoch + 1,
            "train_loss": train_loss, "val_loss": val_loss,
            "is_best": bool(is_best), "epoch_minutes": epoch_minutes,
            "n_train": len(train_records), "n_val": len(val_records),
        })
        # Written every epoch, not just at the end: if the Kaggle session is
        # killed mid-run the curve so far is already on disk.
        pd.DataFrame(history).to_csv(checkpoint_dir / "training_history.csv", index=False)

        print(f"  [{mode}] epoch {epoch+1}/{NUM_EPOCHS}: train={train_loss:.4f} "
              f"val={val_loss:.4f} ({epoch_minutes:.1f} min)" + ("  <- best" if is_best else ""))

        torch.save({
            "epoch": epoch, "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scaler_state": scaler.state_dict() if amp_enabled else None,
            "best_val_loss": best_val_loss, "best_epoch": best_epoch,
            "train_loss": train_loss, "val_loss": val_loss,
            "history": history,
        }, ckpt_path)

    return {"mode": mode, "n_train": len(train_records), "n_val": len(val_records),
            "best_val_loss": best_val_loss, "best_epoch": best_epoch,
            "final_train_loss": history[-1]["train_loss"] if history else float("nan"),
            "final_val_loss": history[-1]["val_loss"] if history else float("nan"),
            "epochs_completed": len(history),
            "best_checkpoint": str(best_path)}, history


# ============================================================================
# CELL 5 — Orchestration
# ============================================================================
def main():
    Path(CHECKPOINT_ROOT).mkdir(parents=True, exist_ok=True)

    print("[1/5] Loading metadata...")
    metadata = pd.read_csv(METADATA_CSV_PATH)
    if isinstance(metadata["labels"].iloc[0], str):
        metadata["labels"] = metadata["labels"].apply(literal_eval)
    train_meta = metadata[metadata["split"] == "train"].reset_index(drop=True)
    print(f"      {len(train_meta)} train images (test split never touched here)")

    print("[2/5] Splitting real train images into train/val (fixed seed)...")
    rng = np.random.default_rng(RANDOM_SEED)
    real_ids = np.array(sorted(train_meta["image_id"].astype(str)))
    rng.shuffle(real_ids)
    n_val = max(1, int(len(real_ids) * VAL_FRACTION))
    val_ids, train_real_ids = real_ids[:n_val].tolist(), real_ids[n_val:].tolist()
    print(f"      {len(train_real_ids)} real train | {len(val_ids)} real val "
          f"(val is REAL-ONLY and identical across all modes)")

    print("[3/5] Building reconstruction manifest...")
    manifest = build_reconstruction_manifest(STAGE1_DIRS_BY_CLASS)

    # A reconstruction of a VAL image must never enter training -- that would
    # leak validation content into the training set through the back door.
    val_set = set(val_ids)
    before = len(manifest)
    manifest = manifest[~manifest["image_id"].isin(val_set)].reset_index(drop=True)
    if before != len(manifest):
        print(f"      excluded {before - len(manifest)} reconstruction(s) of validation images")
    manifest.to_csv(Path(OUTPUT_DIR) / "stage6_reconstruction_manifest.csv", index=False)
    npz_lookup = dict(zip(manifest["image_id"], manifest["npz_path"]))

    val_records = build_records(train_meta, val_ids, "real")

    print("[4/5] Computing class weights (from REAL training images only)...")
    if USE_POS_WEIGHT:
        pos_weight, weight_stats = compute_pos_weight(
            train_meta, train_real_ids, TARGET_CLASSES, POS_WEIGHT_CAP)
        weight_stats.to_csv(Path(OUTPUT_DIR) / f"stage6_pos_weights_{RUN_TAG}.csv", index=False)
        print(weight_stats[["class_name", "n_positives", "prevalence",
                             "raw_pos_weight", "applied_pos_weight", "capped"]].to_string(index=False))
        n_capped = int(weight_stats["capped"].sum())
        if n_capped:
            print(f"      {n_capped} class(es) capped at {POS_WEIGHT_CAP}")
        print("      this SAME vector is used by every arm — never recomputed per-arm")
    else:
        pos_weight = None
        print("      USE_POS_WEIGHT is False — plain unweighted BCE")

    print("[5/5] Training...")
    summaries, all_history = [], []
    for mode in MODES:
        print(f"\n=== mode: {mode}  ({RUN_TAG}) ===")
        aug_ids, description = select_augmentation_ids(manifest, mode, VARIANCE_PERCENTILE, RANDOM_SEED)
        print(f"    {description}")

        records = build_records(train_meta, train_real_ids, "real")
        records += build_records(train_meta, aug_ids, "recon", npz_lookup)
        print(f"    train set: {len(records)} ({len(train_real_ids)} real + {len(aug_ids)} recon)")

        # Record exactly which reconstructions this mode used, so Stage 7 and
        # the write-up can reproduce the arm without re-deriving the selection.
        pd.DataFrame({"image_id": aug_ids}).to_csv(
            Path(CHECKPOINT_ROOT) / f"{mode}_{RUN_TAG}_augmentation_ids.csv", index=False)

        summary, history = train_one_model(
            mode, records, val_records, H5_PATH,
            Path(CHECKPOINT_ROOT) / f"{mode}_{RUN_TAG}", pos_weight=pos_weight
        )
        summary["n_augmentation"] = len(aug_ids)
        summary["augmentation_description"] = description
        summary["run_tag"] = RUN_TAG
        summary["use_pos_weight"] = USE_POS_WEIGHT
        summary["pos_weight_cap"] = POS_WEIGHT_CAP if USE_POS_WEIGHT else None
        summaries.append(summary)
        all_history.extend(history)

    summary_df = pd.DataFrame(summaries)
    out = Path(OUTPUT_DIR) / f"stage6_training_summary_{RUN_TAG}.csv"
    summary_df.to_csv(out, index=False)

    history_df = pd.DataFrame(all_history)
    history_df["run_tag"] = RUN_TAG
    history_out = Path(OUTPUT_DIR) / f"stage6_training_history_{RUN_TAG}.csv"
    history_df.to_csv(history_out, index=False)

    print(f"\nSaved: {out}")
    print(f"Saved: {history_out}  ({len(history_df)} epoch rows across {len(MODES)} mode(s))")
    print(summary_df[["mode", "n_train", "n_augmentation", "epochs_completed",
                       "best_epoch", "best_val_loss", "final_val_loss"]].to_string(index=False))

    print("\nBest checkpoints (for Stage 7):")
    for s in summaries:
        print(f"  {s['mode']:16s} {s['best_checkpoint']}")
    print("\nIMPORTANT: /kaggle/working does NOT persist across sessions. Save this "
          "notebook's output as a Kaggle Dataset before the session ends, or the "
          "checkpoints Stage 7 needs will be lost.")
    print("NOTE: val loss selects checkpoints; it does NOT answer RQ-D. "
          "Run Stage 7 on the untouched test split for that.")
    return summary_df, history_df


if __name__ == "__main__":
    main()

In [ ]:
"""
stage6_replication_driver.py
============================================================================
PASTE AS A CELL *AFTER* the Stage 6 script, in the same notebook.

Runs the two follow-up experiments that turn Stage 7's null into a defensible
result:

  APPROACH C -- SEED REPLICATION
    Stage 7 reported that augmentation degraded Atelectasis (-0.084 AUC) and
    Nodule/Mass (-0.031). With one training run per arm those numbers cannot
    be separated from seed-to-seed training noise. Training `none` and `all`
    at several seeds gives an empirical estimate of that noise, so the
    degradation can be judged against it.

  APPROACH D -- FILTERING-THRESHOLD DOSE-RESPONSE
    Stage 7 tested one filtering threshold (75th percentile) and found no
    benefit over a size-matched random subset. Repeating at the 50th and
    90th percentiles tests whether the null holds across filtering
    aggressiveness. A flat response across three thresholds is much stronger
    evidence than a single null point -- the same dose-response logic that
    made the RQ-C result convincing.

HOW IT WORKS
Stage 6's main() reads module-level globals, so this driver simply reassigns
them and calls main() again. Nothing in Stage 6 needs editing beyond the
TRAINING_SEED separation already applied.

WHAT VARIES, AND WHAT MUST NOT
  varies : TRAINING_SEED (approach C), VARIANCE_PERCENTILE (approach D)
  fixed  : RANDOM_SEED  -> train/val split and random_matched selection stay
                           byte-identical, so every run is trained on the same
                           data and evaluated on the same validation images
           USE_POS_WEIGHT / POS_WEIGHT_CAP -> identical objective throughout
Changing the split alongside the seed would confound training variance with
data variance, which is exactly what approach C is trying to measure.
"""

import itertools
import time

# =============================================================================
# CONFIG
# =============================================================================

# --- Approach C ---------------------------------------------------------
RUN_APPROACH_C = True
SEEDS = [42, 7, 123, 2024, 31337]

# Only the two arms needed to bound the degradation. `filtered` and
# `random_matched` are omitted: Stage 7 already showed they are
# indistinguishable from each other, so replicating both would spend GPU
# hours re-measuring a null.
SEED_MODES = ["none", "all"]

# --- Approach D ---------------------------------------------------------
RUN_APPROACH_D = True
# 75.0 is omitted: it is the run Stage 7 already evaluated. Adding 50 and 90
# gives three points on the dose-response curve.
PERCENTILES = [50.0, 90.0]

# Both arms are needed at each threshold -- `filtered` alone cannot be
# interpreted, because a change could come from the smaller dataset rather
# than from the selection criterion. `random_matched` is size-matched to
# `filtered` at that same percentile, so it moves with it.
PERCENTILE_MODES = ["filtered", "random_matched"]

# Keep these matching the Stage 7 run being extended, or the results are not
# comparable with it.
FIXED_USE_POS_WEIGHT = True
FIXED_POS_WEIGHT_CAP = 20.0


# =============================================================================
def _run(tag, mode_list, training_seed, percentile):
    """One Stage 6 invocation with the given overrides."""
    globals().update({
        "RUN_TAG": tag,
        "MODES": mode_list,
        "TRAINING_SEED": training_seed,
        "VARIANCE_PERCENTILE": percentile,
        "USE_POS_WEIGHT": FIXED_USE_POS_WEIGHT,
        "POS_WEIGHT_CAP": FIXED_POS_WEIGHT_CAP,
    })
    print("\n" + "#" * 74)
    print(f"# RUN_TAG={tag}  modes={mode_list}  seed={training_seed}  pct={percentile}")
    print("#" * 74)
    t0 = time.time()
    main()
    print(f"# {tag} finished in {(time.time() - t0) / 60:.1f} min")


def run_replication():
    planned = []
    if RUN_APPROACH_C:
        planned += [("C", f"weighted-seed{s}", SEED_MODES, s, 75.0) for s in SEEDS]
    if RUN_APPROACH_D:
        planned += [("D", f"weighted-pct{int(p)}", PERCENTILE_MODES, 42, p)
                    for p in PERCENTILES]

    n_models = sum(len(p[2]) for p in planned)
    print(f"Planned: {len(planned)} run(s), {n_models} model(s) total.")
    print(f"At ~20 min/model this is roughly {n_models * 20 / 60:.1f} GPU hours.\n")
    for approach, tag, modes, seed, pct in planned:
        print(f"  [{approach}] {tag:22s} modes={modes} seed={seed} pct={pct}")

    for approach, tag, modes, seed, pct in planned:
        _run(tag, modes, seed, pct)

    print("\n" + "=" * 74)
    print("All replication runs complete. Checkpoints:")
    print("  approach C: <CHECKPOINT_ROOT>/{none,all}_weighted-seed<SEED>/best.pt")
    print("  approach D: <CHECKPOINT_ROOT>/{filtered,random_matched}_weighted-pct<P>/best.pt")
    print("\nNEXT: evaluate them on the test split with stage7_final_evaluation.py.")
    print("  Approach C -> set ARMS=['none','all'] and RUN_TAG='weighted-seed<SEED>',")
    print("               once per seed, then compare the spread of (all - none)")
    print("               across seeds against the -0.084 Atelectasis effect.")
    print("  Approach D -> set ARMS=['filtered','random_matched'] and")
    print("               RUN_TAG='weighted-pct<P>', once per percentile, then plot")
    print("               the filtered-minus-random_matched difference against")
    print("               percentile to read the dose-response.")
    print("=" * 74)


run_replication()